#Data Creation Code

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
!pip install "urllib3<2.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 8.0 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
blobfile 3.2.0 requires urllib3>=2, but you have urllib3 1.26.20 which is incompatible.


In [5]:
#Load in data set using AWS- this is a bigger data source than the synthetic medical data we used in a previous homework.
!apt-get install awscli -q
!aws s3 cp s3://synthea-omop/synthea100k/ ./synthea100k/ --recursive --no-sign-request

Reading package lists...
Building dependency tree...
Reading state information...
awscli is already the newest version (1.22.34-1).
0 upgraded, 0 newly installed, 0 to remove and 6 not upgraded.
/usr/lib/python3/dist-packages/awscli/compat.py:454: SyntaxWarning: invalid escape sequence '\s'
  _distributor_id_file_re = re.compile("(?:DISTRIB_ID\s*=)\s*(.*)", re.I)
/usr/lib/python3/dist-packages/awscli/compat.py:455: SyntaxWarning: invalid escape sequence '\s'
  _release_file_re = re.compile("(?:DISTRIB_RELEASE\s*=)\s*(.*)", re.I)
/usr/lib/python3/dist-packages/awscli/compat.py:456: SyntaxWarning: invalid escape sequence '\s'
  _codename_file_re = re.compile("(?:DISTRIB_CODENAME\s*=)\s*(.*)", re.I)
/usr/lib/python3/dist-packages/awscli/shorthand.py:132: SyntaxWarning: invalid escape sequence '\!'
  _START_WORD = u'\!\#-&\(-\+\--\<\>-Z\\\\-z\u007c-\uffff'
/usr/lib/python3/dist-packages/awscli/shorthand.py:133: SyntaxWarning: invalid escape sequence '\s'
  _FIRST_FOLLOW_CHARS = u'\s\!\#-&\

In [6]:
!apt-get install -y lzop
!lzop -d synthea100k/*.lzo

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  lzop
0 upgraded, 1 newly installed, 0 to remove and 6 not upgraded.
Need to get 83.7 kB of archives.
After this operation, 163 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 lzop amd64 1.04-2build2 [83.7 kB]
Fetched 83.7 kB in 1s (66.7 kB/s)
Selecting previously unselected package lzop.
(Reading database ... 129222 files and directories currently installed.)
Preparing to unpack .../lzop_1.04-2build2_amd64.deb ...
Unpacking lzop (1.04-2build2) ...
Setting up lzop (1.04-2build2) ...
Processing triggers for man-db (2.10.2-1) ...


##Creating the 4 relational tables from the AWS data source

In [7]:
person = pd.read_csv("synthea100k/person.csv")
person_clean = person.drop_duplicates(subset=['person_id'])

person_table = person_clean[['person_id', 'gender_concept_id', 'race_concept_id', 'birth_datetime']]

In [8]:
visits = pd.read_csv("synthea100k/visit_occurrence.csv")

visit_table = visits[['visit_occurrence_id', 'person_id', 'visit_concept_id', 'visit_start_date']]

In [9]:
hospital_visits = visit_table[visit_table['visit_concept_id'] == 9201].copy()

hospital_visits['visit_start_date'] = pd.to_datetime(hospital_visits['visit_start_date'])

hospital_visits = hospital_visits.sort_values(['person_id', 'visit_start_date'])

hospital_visits['next_visit_start'] = hospital_visits.groupby('person_id')['visit_start_date'].shift(-1)

hospital_visits['DAYS_TO_NEXT'] = (
    hospital_visits['next_visit_start'] - hospital_visits['visit_start_date']
).dt.days

hospital_visits['READMIT_30'] = hospital_visits['DAYS_TO_NEXT'].apply(
    lambda x: 1 if pd.notnull(x) and x <= 30 else 0
)

In [10]:
data = hospital_visits.merge(person_table, on='person_id', how='left')

data['AGE'] = (
    pd.to_datetime('today') - pd.to_datetime(data['birth_datetime'])
).dt.days // 365

data = data.sort_values(['person_id', 'visit_start_date'])

data['PRIOR_ADMISSIONS'] = data.groupby('person_id').cumcount()

readmission_features = data[
    ['person_id', 'visit_occurrence_id', 'READMIT_30', 'AGE', 'gender_concept_id', 'race_concept_id', 'PRIOR_ADMISSIONS']
]

##Save data as csv files and as parquet

In [11]:
# Create an output folder
import os
os.makedirs("output_data", exist_ok=True)

# Save CSVs
person_table.to_csv("output_data/person.csv", index=False)
visit_table.to_csv("output_data/visit_occurrence.csv", index=False)
hospital_visits.to_csv("output_data/hospital_visits.csv", index=False)
readmission_features.to_csv("output_data/readmission_features.csv", index=False)

# Save Parquet files
person_table.to_parquet("output_data/person.parquet", index=False)
visit_table.to_parquet("output_data/visit_occurrence.parquet", index=False)
hospital_visits.to_parquet("output_data/hospital_visits.parquet", index=False)
readmission_features.to_parquet("output_data/readmission_features.parquet", index=False)


In [12]:
import shutil

shutil.make_archive("output_data", 'zip', "output_data")
#After putting it into a zip file, download to your local computer and upload to One Drive Folder

'/content/output_data.zip'